# 05b · Training Craft `[local]`

**Goal:** learn the techniques that turn a mediocre training run into a good one, and the habit of proving each one helps with a logged experiment rather than assuming it does.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | Why the recipe matters as much as the model |
| 📐 Math | MixUp/CutMix, label smoothing, weight decay, LR schedules, mixed precision, gradient accumulation, imbalance losses, metrics |
| 🔨 From scratch | LR finder, MixUp/CutMix, focal loss, gradient accumulation check, precision/recall/F1/AP |
| 🛠 Project | **Push a weak CIFAR-10 baseline as high as you can**, one logged change at a time |
| ⚖️ Imbalance | Long-tailed CIFAR: weighted loss vs. focal loss vs. oversampling |
| 📊 Evaluate | Macro vs. micro metrics, per-class recall, PR curves |
| 🧪 Experiment | Mixed-precision speed, label smoothing and confidence, warmup |

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import copy
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from cvlib.datasets import cifar10, CIFAR10_CLASSES
from cvlib.loaders import GPULoader, random_crop_flip, CIFAR_MEAN, CIFAR_STD
from cvlib.models import SimpleCNN
from cvlib.tracking import ExperimentLog
from cvlib import train as cvtrain
from cvlib.viz import plot_history

device = env.device
rng = np.random.default_rng(0)

X_all, y_all = cifar10("train")
perm = rng.permutation(len(X_all))
X_train, y_train = X_all[perm[:45_000]], y_all[perm[:45_000]]
X_val, y_val = X_all[perm[45_000:]], y_all[perm[45_000:]]
X_test, y_test = cifar10("test")
X_sub, y_sub = X_train[:20_000], y_train[:20_000]                      # a 20k subset keeps each run ~1 minute
print(f"train {len(y_train)} | subset {len(y_sub)} | val {len(y_val)} | test {len(y_test)}")

def loader(X, y, train=True, aug=True, bs=128):
    return GPULoader(X, y, batch_size=bs if train else 1000, shuffle=train, augment=random_crop_flip if (train and aug) else None,
                     mean=CIFAR_MEAN, std=CIFAR_STD, device=device)

val_loader = loader(X_val, y_val, train=False)

---
# 🧠 1 · Intuition

The same architecture can land anywhere from mediocre to state-of-the-art depending on **how it's trained**. The recipe has a few families of knobs:

- **Data:** augmentation creates new, realistic variations, so the model can't memorize. MixUp/CutMix blend images, so it can't be overconfident.
- **Regularization:** weight decay keeps weights small; label smoothing stops the model from chasing 100% confidence.
- **Optimization:** the learning rate and how it changes over time (schedule, warmup) decide whether training is fast, stable, and ends in a good spot.
- **Efficiency:** mixed precision and gradient accumulation let you train bigger models or batches on the same hardware.
- **The data's shape:** with imbalanced classes, plain training ignores the rare ones, and plain accuracy hides it.

**The rule of this notebook:** change **one thing at a time**, measure it on validation data, log it. Intuition about what "should" help is often wrong.

---
# 📐 2 · The math

### MixUp and CutMix
Pick two training images $(x_i, y_i)$, $(x_j, y_j)$ and $\lambda \sim \text{Beta}(\alpha, \alpha)$:
- **MixUp:** $\tilde{x} = \lambda x_i + (1-\lambda) x_j$, a see-through blend
- **CutMix:** paste a box from $x_j$ into $x_i$, covering a fraction $1-\lambda$ of the area
- Both: the label is mixed too, $\tilde{y} = \lambda\, y_i + (1-\lambda)\, y_j$ (one-hot vectors). The model must say "70% dog, 30% cat".

### Label smoothing
Instead of a hard target (1 for the right class, 0 elsewhere), use $1 - \varepsilon$ for the right class and $\varepsilon / (K-1)$ for each other class (PyTorch spreads $\varepsilon / K$ over all classes, a near-identical variant). The model is no longer rewarded for pushing logits to infinity, which makes it less overconfident.

### Weight decay, and why AdamW
L2 regularization adds $\frac{\lambda}{2}\|w\|^2$ to the loss. With SGD that's the same as shrinking weights a little every step ("decay"). With Adam it isn't, because Adam rescales gradients per weight. **AdamW** applies the decay directly to the weights, separately from the gradient step. Use AdamW, not Adam + L2.

### Learning-rate schedules
With $t$ = step, $T$ = total steps:
- **Step decay:** multiply by 0.1 at fixed epochs
- **Cosine:** $\eta_t = \eta_{\min} + \tfrac{1}{2}(\eta_{\max} - \eta_{\min})(1 + \cos(\pi t / T))$
- **Warmup:** start small and rise linearly for the first few % of steps. Early gradients are large and chaotic; warmup prevents an early blow-up.
- **OneCycle:** warmup to a high max LR, then anneal to near zero. Often the fastest-converging choice.

### Mixed precision (AMP)
float16 uses half the memory and runs on fast tensor cores, but it only covers about $6\times10^{-8}$ to $65{,}504$. Tiny gradients **underflow** to zero, so CUDA training multiplies the loss by a large factor $S$ before backprop and divides gradients by $S$ afterwards (`GradScaler`). Precision-sensitive ops (softmax, losses) stay in float32 automatically.

### Gradient accumulation
To mimic batch $B = k \cdot b$ when only $b$ fits in memory: run $k$ micro-batches, divide each loss by $k$, call `backward()` on each (gradients add up), then one `optimizer.step()`. Exactly equal to the big batch, **except** for BatchNorm, which still only sees $b$ images at a time.

### Imbalanced classes
With class counts $n_c$:
- **Weighted cross-entropy:** weight each class's loss by $w_c \propto 1/n_c$
- **Focal loss:** $\text{FL} = -(1 - p_t)^\gamma \log p_t$, which down-weights easy, well-classified examples (usually from the big classes) so training focuses on the hard ones. $\gamma = 0$ gives plain cross-entropy.
- **Oversampling:** show rare-class images more often

### Precision, recall, F1: macro vs. micro
For class $c$: precision $P_c = \frac{TP_c}{TP_c + FP_c}$, recall $R_c = \frac{TP_c}{TP_c + FN_c}$, $F1_c = \frac{2 P_c R_c}{P_c + R_c}$.
- **Macro** average: mean over classes. Every class counts equally, so rare classes matter.
- **Micro** average: pool all decisions first. For single-label classification, micro-F1 = **accuracy**, dominated by the big classes.

---
# 🔨 3 · From scratch

### 3.1 · The learning-rate finder

Train for ~100 batches while increasing the LR exponentially from $10^{-6}$ to $1$, recording the loss. The loss falls, flattens, then explodes. A good max LR sits a bit **before** the bottom, where the loss is still dropping steeply. (Leslie Smith, 2015; popularized by fast.ai.)

In [ ]:
def lr_finder(make_model, loader, lr_min=1e-6, lr_max=1.0, steps=100, momentum=0.9):
    model = make_model().to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr_min, momentum=momentum)
    mult = (lr_max / lr_min) ** (1 / steps)
    lrs, losses, avg, best = [], [], 0.0, float("inf")
    it = iter(loader)
    model.train()
    for i in range(steps):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(loader); x, y = next(it)
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
        avg = 0.9 * avg + 0.1 * loss.item()                          # exponential smoothing
        smooth = avg / (1 - 0.9 ** (i + 1))                           # bias correction for early steps
        lrs.append(opt.param_groups[0]["lr"]); losses.append(smooth)
        best = min(best, smooth)
        if smooth > 4 * best:                                         # exploded: stop
            break
        opt.param_groups[0]["lr"] *= mult
    return np.array(lrs), np.array(losses)

torch.manual_seed(0)
lrs, losses = lr_finder(SimpleCNN, loader(X_sub, y_sub))
steepest = lrs[np.argmin(np.gradient(losses, np.log(lrs)))]
plt.figure(figsize=(7, 3.5))
plt.plot(lrs, losses); plt.xscale("log"); plt.xlabel("learning rate"); plt.ylabel("smoothed loss")
plt.axvline(steepest, color="r", ls="--", label=f"steepest descent ≈ {steepest:.1e}")
plt.axvline(lrs[np.argmin(losses)], color="gray", ls=":", label=f"minimum ≈ {lrs[np.argmin(losses)]:.1e}")
plt.legend(); plt.grid(alpha=0.3); plt.title("LR finder (SGD + momentum, SimpleCNN)"); plt.show()

### 3.2 · MixUp and CutMix

They need to change the **labels** too, so they wrap a loader: hard labels in, mixed images and soft labels out. `nn.CrossEntropyLoss` accepts probability targets directly.

In [ ]:
class MixLoader:
    def __init__(self, base, n_classes=10, mode="mixup", alpha=1.0, p=0.5):
        self.base, self.n, self.mode, self.alpha, self.p = base, n_classes, mode, alpha, p

    def __len__(self):
        return len(self.base)

    def __iter__(self):
        for x, y in self.base:
            y1 = F.one_hot(y, self.n).float()
            if np.random.rand() > self.p:                              # mix only some batches
                yield x, y1; continue
            lam = float(np.random.beta(self.alpha, self.alpha))
            j = torch.randperm(len(x), device=x.device)
            if self.mode == "mixup":
                x = lam * x + (1 - lam) * x[j]
            else:                                                      # cutmix: paste a box from the partner image
                H, W = x.shape[-2:]
                ch, cw = int(H * np.sqrt(1 - lam)), int(W * np.sqrt(1 - lam))
                cy, cx = np.random.randint(H), np.random.randint(W)
                y0, y1_, x0, x1 = max(cy - ch // 2, 0), min(cy + ch // 2, H), max(cx - cw // 2, 0), min(cx + cw // 2, W)
                x = x.clone(); x[:, :, y0:y1_, x0:x1] = x[j, :, y0:y1_, x0:x1]
                lam = 1 - (y1_ - y0) * (x1 - x0) / (H * W)             # actual uncovered area
            yield x, lam * y1 + (1 - lam) * y1[j]

demo = loader(X_sub[:8], y_sub[:8], aug=False, bs=8)
fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for row, mode in enumerate(["mixup", "cutmix"]):
    np.random.seed(3); torch.manual_seed(3)
    xm, ym = next(iter(MixLoader(demo, mode=mode, p=1.0)))
    xm = xm * torch.tensor(CIFAR_STD, device=device).view(1, 3, 1, 1) + torch.tensor(CIFAR_MEAN, device=device).view(1, 3, 1, 1)
    for k in range(6):
        top2 = torch.topk(ym[k], 2)
        axes[row, k].imshow(xm[k].clamp(0, 1).permute(1, 2, 0).cpu()); axes[row, k].axis("off")
        axes[row, k].set_title(" + ".join(f"{CIFAR10_CLASSES[i][:4]} {v:.0%}" for v, i in zip(top2.values.tolist(), top2.indices.tolist()) if v > 0.01), fontsize=7)
    axes[row, 0].set_ylabel(mode)
plt.suptitle("top: MixUp | bottom: CutMix (titles = soft labels)"); plt.tight_layout(); plt.show()

### 3.3 · Focal loss and label smoothing, checked against PyTorch

In [ ]:
def focal_loss(logits, y, gamma=2.0, weight=None):
    logp = F.log_softmax(logits, 1)
    logp_t = logp.gather(1, y[:, None]).squeeze(1)                    # log-probability of the true class
    p_t = logp_t.exp()
    loss = -((1 - p_t) ** gamma) * logp_t                              # easy examples (p_t → 1) contribute ~0
    if weight is not None:
        loss = loss * weight[y]
    return loss.mean()

def smooth_ce(logits, y, eps=0.1):
    K = logits.shape[1]
    target = torch.full_like(logits, eps / K).scatter_(1, y[:, None], 1 - eps + eps / K)   # PyTorch's convention
    return -(target * F.log_softmax(logits, 1)).sum(1).mean()

lg, yy = torch.randn(64, 10), torch.randint(0, 10, (64,))
print("focal(γ=0) == cross-entropy :", torch.allclose(focal_loss(lg, yy, gamma=0), F.cross_entropy(lg, yy)))
print("smooth_ce == PyTorch's      :", torch.allclose(smooth_ce(lg, yy), F.cross_entropy(lg, yy, label_smoothing=0.1)))

### 3.4 · Gradient accumulation is exact (without BatchNorm)

In [ ]:
def grads_after(model, batches, scale):
    model.zero_grad()
    for x, y in batches:
        (F.cross_entropy(model(x), y) * scale).backward()             # gradients accumulate across calls
    return torch.cat([p.grad.flatten() for p in model.parameters()])

xb, yb = next(iter(loader(X_sub, y_sub, aug=False, bs=64)))
for bn in (False, True):
    torch.manual_seed(0); m = SimpleCNN(bn=bn).to(device).train()
    g_big = grads_after(m, [(xb, yb)], 1.0)
    g_acc = grads_after(m, [(xb[i:i + 16], yb[i:i + 16]) for i in range(0, 64, 16)], 1 / 4)
    print(f"BatchNorm={bn!s:<5}: max |grad(batch 64) − grad(4 × 16 accumulated)| = {(g_big - g_acc).abs().max():.2e}")

Without BatchNorm the two match to rounding error. With BatchNorm they differ, because each micro-batch is normalized with its own statistics. That's the one caveat of accumulation.

---
# 🛠 4 · Project: push a weak baseline, one logged change at a time

Same model (module 03's `SimpleCNN`), same 20k-image subset, same 8 epochs. Each run adds **one** change. Every result goes into the experiment log.

In [ ]:
log = ExperimentLog(env.out_dir / "cifar_recipe_runs.csv", reset=True)

def recipe_run(name, aug=False, opt_name="sgd", lr=0.05, wd=0.0, schedule="constant", smoothing=0.0, mix=None, epochs=8,
               X=X_sub, y=y_sub, log_it=True):
    torch.manual_seed(0); np.random.seed(0)
    model = SimpleCNN().to(device)
    tl = loader(X, y, aug=aug)
    if mix:
        tl = MixLoader(tl, mode=mix, p=0.5)
    if opt_name == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd, nesterov=True)
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = batch_sched = None
    if schedule == "cosine":
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    elif schedule == "onecycle":
        batch_sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(tl), pct_start=0.15)
    t0 = time.perf_counter()
    h = cvtrain.fit(model, tl, val_loader, nn.CrossEntropyLoss(label_smoothing=smoothing), opt, epochs=epochs, device=device,
                    scheduler=sched, batch_scheduler=batch_sched, verbose=False)
    mins = (time.perf_counter() - t0) / 60
    if log_it:
        log.add(name=name, aug=aug, opt=opt_name, lr=lr, wd=wd, schedule=schedule, smoothing=smoothing, mix=mix or "-",
                epochs=epochs, n_train=len(y), val_acc=round(h["val_acc"][-1], 4), val_loss=round(h["val_loss"][-1], 4),
                train_loss=round(h["train_loss"][-1], 4), minutes=round(mins, 2))
    print(f"{name:<34} val acc {h['val_acc'][-1]:.4f} | train loss {h['train_loss'][-1]:.3f} | val loss {h['val_loss'][-1]:.3f} | {mins:.1f} min")
    return model, h

runs = {}
runs["1 baseline"] = recipe_run("1 baseline: SGD, const LR")
runs["2 +aug"] = recipe_run("2 + crop/flip augmentation", aug=True)
runs["3 +onecycle"] = recipe_run("3 + OneCycle (max lr 0.1)", aug=True, lr=0.1, schedule="onecycle")
runs["4 +wd"] = recipe_run("4 + weight decay 5e-4", aug=True, lr=0.1, schedule="onecycle", wd=5e-4)
runs["5 +ls"] = recipe_run("5 + label smoothing 0.1", aug=True, lr=0.1, schedule="onecycle", wd=5e-4, smoothing=0.1)
runs["6 +cutmix"] = recipe_run("6 + CutMix (half the batches)", aug=True, lr=0.1, schedule="onecycle", wd=5e-4, smoothing=0.1, mix="cutmix")
log.table()[["name", "val_acc", "val_loss", "train_loss", "minutes"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for name, (_, h) in runs.items():
    axes[0].plot(h["val_acc"], marker="o", ms=3, label=name)
    axes[1].plot(h["train_loss"], label=name)
axes[0].set_title("validation accuracy"); axes[1].set_title("training loss")
for a in axes: a.set_xlabel("epoch"); a.grid(alpha=0.3); a.legend(fontsize=7)
plt.tight_layout(); plt.show()

**Reading the log:**
- **Augmentation** raises *training* loss (the task got harder) while improving validation: that's regularization doing its job. Without it, the baseline memorizes.
- **OneCycle** usually gives the biggest single jump at a fixed epoch budget: high LR explores, the long anneal settles into a good minimum.
- **Weight decay** and **label smoothing** are smaller, steadier gains. Label smoothing raises the training loss floor by design.
- **CutMix** often *hurts* in a short 8-epoch run: it makes training harder, and its payoff shows up in long runs (100+ epochs). A good reminder that a technique's value depends on the budget.

Whatever your exact numbers are, the log tells you which changes earned their place.

### 4.2 · Final: the best recipe on all 45k images, longer

Choose the recipe by **validation** accuracy, train it on the full training set for longer, then evaluate on **test once**.

In [ ]:
best_name = log.table(sort_by="val_acc").iloc[0]["name"]
best_row = log.table(sort_by="val_acc").iloc[0]
print("best recipe by validation:", best_name)
final_model, h_final = recipe_run("FINAL: best recipe, 45k images, 20 epochs", aug=bool(best_row["aug"]), lr=float(best_row["lr"]),
                                  wd=float(best_row["wd"]), schedule=best_row["schedule"], smoothing=float(best_row["smoothing"]),
                                  mix=None if best_row["mix"] == "-" else best_row["mix"], epochs=20, X=X_train, y=y_train)
test_loader = loader(X_test, y_test, train=False)
_, test_acc = cvtrain.evaluate(final_model, test_loader, nn.CrossEntropyLoss(), device)
print(f"\nTEST accuracy: {test_acc:.4f}   (module 03's run: ~0.88)")

---
# ⚖️ 5 · Class imbalance: long-tailed CIFAR

Real datasets are rarely balanced: defect detection, medical findings, document types. We build a **long-tailed** CIFAR: class 0 keeps 4,500 images, class 9 only 90 (50× fewer), decaying exponentially in between. Validation stays balanced, so every class counts equally.

In [ ]:
imb_ratio = 50
counts = [int(4500 * (1 / imb_ratio) ** (c / 9)) for c in range(10)]
lt_idx = np.concatenate([np.nonzero(y_train == c)[0][:n] for c, n in enumerate(counts)])
X_lt, y_lt = X_train[lt_idx], y_train[lt_idx]
plt.figure(figsize=(8, 2.8)); plt.bar(CIFAR10_CLASSES, counts); plt.ylabel("training images"); plt.xticks(rotation=30)
plt.title(f"Long-tailed CIFAR-10: {len(y_lt):,} images, imbalance ratio {imb_ratio}"); plt.tight_layout(); plt.show()

In [ ]:
def per_class_metrics(y_true, y_pred, n=10):
    cm = np.zeros((n, n), int); np.add.at(cm, (y_true, y_pred), 1)
    tp = cm.diagonal().astype(float)
    precision = tp / np.maximum(cm.sum(0), 1)
    recall = tp / np.maximum(cm.sum(1), 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-9)
    return dict(accuracy=tp.sum() / cm.sum(), macro_f1=f1.mean(), micro_f1=tp.sum() / cm.sum(), recall=recall, precision=precision, f1=f1)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__(); self.gamma, self.weight = gamma, weight
    def forward(self, logits, y):
        return focal_loss(logits, y, self.gamma, self.weight)

w = torch.tensor([1 / c for c in counts], dtype=torch.float32, device=device)
w = w / w.mean()                                                        # weights average to 1
over_idx = np.concatenate([np.random.default_rng(c).choice(np.nonzero(y_lt == c)[0], counts[0], replace=True) for c in range(10)])

methods = {
    "plain cross-entropy": (nn.CrossEntropyLoss(), X_lt, y_lt),
    "class-weighted CE": (nn.CrossEntropyLoss(weight=w), X_lt, y_lt),
    "focal loss (γ=2)": (FocalLoss(2.0), X_lt, y_lt),
    "oversampling": (nn.CrossEntropyLoss(), X_lt[over_idx], y_lt[over_idx]),
}
imb_results = {}
for name, (loss_fn, Xm, ym) in methods.items():
    torch.manual_seed(0)
    m = SimpleCNN().to(device)
    tl = loader(Xm, ym)
    steps = 8 * len(loader(X_lt, y_lt))                                  # same number of updates for every method
    epochs = max(1, round(steps / len(tl)))
    opt = torch.optim.SGD(m.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=0.1, total_steps=epochs * len(tl), pct_start=0.15)
    cvtrain.fit(m, tl, val_loader, loss_fn, opt, epochs=epochs, device=device, batch_scheduler=sched, verbose=False)
    logits, labels = cvtrain.predict(m, val_loader, device)
    imb_results[name] = per_class_metrics(labels.numpy(), logits.argmax(1).numpy())
    r = imb_results[name]
    print(f"{name:<22} accuracy {r['accuracy']:.3f} | macro-F1 {r['macro_f1']:.3f} | rarest class recall {r['recall'][9]:.3f} | worst class recall {r['recall'].min():.3f}")

---
# 📊 6 · Evaluate: what accuracy hides

### 6.1 · Per-class recall: head vs. tail

In [ ]:
x_pos = np.arange(10); wbar = 0.2
plt.figure(figsize=(11, 3.8))
for k, (name, r) in enumerate(imb_results.items()):
    plt.bar(x_pos + (k - 1.5) * wbar, r["recall"], wbar, label=name)
plt.xticks(x_pos, [f"{c}\n({n})" for c, n in zip(CIFAR10_CLASSES, counts)], fontsize=8)
plt.ylabel("recall on balanced validation"); plt.title("Per-class recall: class name (training images)")
plt.legend(fontsize=8); plt.grid(alpha=0.3, axis="y"); plt.tight_layout(); plt.show()

With plain cross-entropy, the head classes (airplane, automobile) have high recall and the tail classes (ship, truck) collapse. The model learned that "when unsure, guess a common class" pays off. The rebalancing methods trade a little head-class recall for much better tail recall, which shows up in **macro-F1** much more than in accuracy.

**Focal loss alone barely helps here**, and that's expected. It down-weights *easy* examples, not *frequent classes*; it was designed for detection (module 06), where the imbalance is millions of easy background boxes vs. a few objects. For class-frequency imbalance, class weights or resampling are the direct fix (focal loss can be combined with class weights).

**Which number to report?** Accuracy (= micro-F1 here) if every *image* matters equally. Macro-F1 or per-class recall if every *class* matters, e.g. a rare defect type you must not miss.

### 6.2 · Precision-recall curve and average precision, from scratch

For one class, sweep the decision threshold on its predicted probability: each threshold gives one (recall, precision) point. **Average precision (AP)** summarizes the curve. AP and its mean over classes (**mAP**) are *the* detection metric in module 06, so it's worth building now.

In [ ]:
def pr_curve(scores, positives):
    order = np.argsort(-scores)
    tp = np.cumsum(positives[order]); fp = np.cumsum(~positives[order])
    precision = tp / (tp + fp); recall = tp / positives.sum()
    return precision, recall

def average_precision(scores, positives):
    precision, recall = pr_curve(scores, positives)
    precision = np.maximum.accumulate(precision[::-1])[::-1]               # make precision non-increasing (interpolated)
    r_prev = np.concatenate([[0], recall[:-1]])
    return float(np.sum((recall - r_prev) * precision))

torch.manual_seed(0)
plain = SimpleCNN().to(device)
opt = torch.optim.SGD(plain.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
tl = loader(X_lt, y_lt)
cvtrain.fit(plain, tl, val_loader, nn.CrossEntropyLoss(), opt, epochs=8, device=device,
            batch_scheduler=torch.optim.lr_scheduler.OneCycleLR(opt, 0.1, total_steps=8 * len(tl), pct_start=0.15), verbose=False)
probs, labels = cvtrain.predict(plain, val_loader, device)
probs = F.softmax(probs, 1).numpy(); labels = labels.numpy()

plt.figure(figsize=(6, 4.5))
for c in (0, 4, 9):
    p, r = pr_curve(probs[:, c], labels == c)
    plt.plot(r, p, label=f"{CIFAR10_CLASSES[c]} ({counts[c]} train imgs): AP {average_precision(probs[:, c], labels == c):.3f}")
plt.xlabel("recall"); plt.ylabel("precision"); plt.title("PR curves (plain CE on long-tailed data)"); plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()
print(f"mean AP over all 10 classes: {np.mean([average_precision(probs[:, c], labels == c) for c in range(10)]):.3f}")

AP is **threshold-free**: it measures how well the model *ranks* positives above negatives, regardless of where you'd put the cut-off. A rare class can have decent AP even when its recall at the default argmax decision is poor. Rebalancing or **choosing a per-class threshold** could recover a lot of it, which is exactly what 05c does for multi-label tagging.

---
# 🧪 7 · Experiments: predict first, then run

**Experiment 1: does mixed precision speed things up here?** Predict for your Mac's GPU, then for an NVIDIA GPU.

In [ ]:
# Prediction: ...
for amp in (False, True):
    torch.manual_seed(0)
    m = SimpleCNN().to(device)
    o = torch.optim.SGD(m.parameters(), lr=0.05, momentum=0.9)
    tl = loader(X_sub, y_sub)
    cvtrain.train_one_epoch(m, tl, nn.CrossEntropyLoss(), o, device, amp=amp)        # warm-up epoch
    t0 = time.perf_counter()
    scaler = torch.amp.GradScaler("cuda") if (amp and device == "cuda") else None
    loss, acc = cvtrain.train_one_epoch(m, tl, nn.CrossEntropyLoss(), o, device, amp=amp, scaler=scaler)
    print(f"amp={amp!s:<5} on {device}: {time.perf_counter() - t0:.1f}s/epoch | train acc {acc:.3f}")
# On NVIDIA GPUs with tensor cores (T4, A100, ...) AMP is typically 1.5–3× faster and halves activation memory.
# On Apple GPUs (MPS) float16 autocast is often *not* faster, sometimes slower. Measure on your hardware.
# Module 15 goes deeper on precision formats.

**Experiment 2: label smoothing and confidence.** Predict how the average confidence on validation images changes with smoothing 0.1, compared to none.

In [ ]:
# Prediction: ...
for label, key in [("no smoothing (run 4)", "4 +wd"), ("smoothing 0.1 (run 5)", "5 +ls")]:
    lg, lb = cvtrain.predict(runs[key][0], val_loader, device)
    p = F.softmax(lg, 1)
    correct = p.argmax(1) == lb
    print(f"{label:<22}: mean confidence {p.max(1).values.mean():.3f} | when right {p.max(1).values[correct].mean():.3f} | "
          f"when wrong {p.max(1).values[~correct].mean():.3f} | accuracy {correct.float().mean():.3f}")
# Smoothing lowers confidence, especially on mistakes, making the model less overconfident.
# Module 14 measures this properly (calibration).

**Experiment 3: warmup and a high learning rate.** Predict: at max LR 0.4, does skipping warmup matter?

In [ ]:
# Prediction: ...
for pct in (0.01, 0.15):
    torch.manual_seed(0)
    m = SimpleCNN().to(device)
    tl = loader(X_sub, y_sub)
    o = torch.optim.SGD(m.parameters(), lr=0.4, momentum=0.9, nesterov=True)
    s = torch.optim.lr_scheduler.OneCycleLR(o, max_lr=0.4, total_steps=4 * len(tl), pct_start=pct)
    h = cvtrain.fit(m, tl, val_loader, nn.CrossEntropyLoss(), o, epochs=4, device=device, batch_scheduler=s, verbose=False)
    print(f"warmup over {pct:.0%} of steps: val acc per epoch {[round(a, 3) for a in h['val_acc']]}")
# Even this small BatchNorm CNN learns much worse when it jumps straight to a high LR: early updates are large and
# chaotic and knock the weights into a bad region. Warmup matters even more for large models, big batches,
# Adam-type optimizers and transformers (module 08), where skipping it can make training diverge entirely.

---
# 📝 8 · Recap

**Cheat sheet**
- Change **one thing at a time**, judge on **validation**, log every run. Test once at the end.
- **LR finder** → pick a max LR on the steep part. **OneCycle / cosine + warmup** are strong defaults.
- **Augmentation** (crop/flip) is the biggest cheap win; **MixUp/CutMix** pay off in long runs.
- **AdamW** (decoupled weight decay), not Adam + L2. **Label smoothing** reduces overconfidence.
- **AMP**: float16 + loss scaling. Big win on NVIDIA; measure elsewhere.
- **Gradient accumulation** = bigger effective batch, exact except for BatchNorm.
- **Imbalance**: weighted CE, focal loss, oversampling. Report **macro-F1 / per-class recall**, not just accuracy.
- **AP** = area under the precision-recall curve: threshold-free ranking quality. **mAP** = mean over classes.

**Interview-style questions**
1. How does an LR finder work, and how do you read its plot?
2. Why does augmentation increase training loss but improve validation accuracy?
3. What's the difference between Adam with L2 regularization and AdamW?
4. Why does mixed-precision training need loss scaling? Which operations stay in float32?
5. Gradient accumulation vs. a real large batch: when are they not equivalent?
6. Your fraud model has 99.5% accuracy on data with 0.5% fraud. Is it good? What would you report instead?
7. Explain focal loss. What does γ control?
8. Macro vs. micro F1: when does each make sense?

**Next:** `05c_multilabel.ipynb`, when an image can have several labels at once.